In [1]:
from mygenerator import MyGenerator
import matplotlib.pyplot as plt
#from spectrogramgenerator import spectrogram
import random
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import scipy

fnames = glob("/tmp/foo/sancsound/clips/train/*npy")
N = 256  # we want a spectrogram of this dim
RESAMPLE=10

In [2]:
def getXSpec():
    f = random.choice(fnames)
    N = 256
    x = np.load( f)
    x = scipy.signal.resample(x,x.shape[0]//RESAMPLE)

    i = random.choice(range(0, x.shape[0] -(N//2)**2))
    x=x[i:i+(N//2+1)*(N//2 - 1)]
    _,_,spec =  scipy.signal.stft(x, nperseg=N)
    spec = np.log(np.abs(spec))
    x = np.expand_dims(x,-1)
    spec= np.expand_dims(spec, -1)
    return x, spec

def getABC():
    x1, spec1= getXSpec()
    x2, spec2= getXSpec()
    a = x1
    b = x2
    choice = random.choice([0,1])
    if (choice==0):
        c = spec1
    else:
        c= spec2
    y = choice
    return [np.expand_dims(x1,0), np.expand_dims(x2, 0), np.expand_dims(c,0)], np.array([y])



In [3]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam


def build1D():
    inp = Input(shape=(16383,1))
    x = Conv1D(16,5, activation='relu', padding='same')(inp)
    x = Conv1D(32, 3, activation='relu', padding='same')(x)
    x = Conv1D(64, 3, activation='relu', )(x)
    x = Flatten()(x)
    return Model(inp, x)

def build2D():
    inp = Input(shape=(129,129,1))
    x = Conv2D(64,5, activation='relu', padding='same')(inp)
    x = Conv2D(64, 3, activation='relu', padding='same')(x)
    x = Conv2D(128, 3, activation='relu',)(x)
    x = MaxPooling2D()(x)
    x = Flatten()(x)
    return Model(inp, x)

m1 = build1D()
m2 = build1D()
m3 = build2D()

c = concatenate([m1.output, m2.output, m3.output])
d = Dense(60, activation='relu')(c)
d = Dense(10, activation='relu')(d)
d = Dense(1, activation='sigmoid')(d)
model = Model([m1.input, m2.input, m3.input], d)
model.summary()
#m3.summary()

Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 129, 129, 1  0           []                               
                                )]                                                                
                                                                                                  
 input_1 (InputLayer)           [(None, 16383, 1)]   0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, 16383, 1)]   0           []                               
                                                                                                  
 conv2d (Conv2D)                (None, 129, 129, 64  1664        ['input_3[0][0]']          

In [4]:
model.compile(loss = 'binary_crossentropy', optimizer=Adam(learning_rate=0.0003), metrics=['acc'])

In [ ]:
history = []
for i in range(2000000):
    x, y = getABC()
    h = model.fit(x,y, verbose=0)
    if (i%500 ==0 and i>0):
        print(h.history['loss'])
         
    history.append(h)

[0.6884347200393677]
[0.6948376297950745]
[0.7011587023735046]
[0.6750925183296204]
[0.7160475254058838]
[0.6750837564468384]
[0.6817163228988647]
[0.6873233914375305]
[0.6870855093002319]
[0.6985417008399963]
[0.6946446895599365]
[0.6962429285049438]
[0.6901233792304993]
[0.6975194811820984]
[0.6968778371810913]
[0.6886625289916992]
[0.691433846950531]
[0.6923120617866516]
[0.6905016303062439]
[0.7076910734176636]
[0.697550892829895]
[0.6899150013923645]
[0.685700535774231]
[0.6973021030426025]
[0.6818028688430786]
[0.6686805486679077]
[0.7098791599273682]
[0.7199511528015137]
[0.6817265748977661]
[0.6935677528381348]
[0.6943281888961792]
[0.693971574306488]
[0.6983950734138489]
[0.6812118887901306]
[0.7109278440475464]
[0.7098476886749268]
[0.6803253889083862]
[0.7021185159683228]
[0.8700829148292542]


In [ ]:
x = [h.history['loss'][0] for h in history] 
x = np.array(x)
x=x-x.mean()
plt.plot(np.log(x));

In [ ]:
v= np.diff(x)
a = np.diff(v)

In [ ]:
plt.scatter(x[1:],v)

In [ ]:
ka=[]
kb=[]
for a,b in zip(x[1:],v):
    if(np.abs(b)<5):
        ka.append(a)
        kb.append(b)
plt.scatter(ka,kb,s=1)     